# 第 5 周 · 周一：SQL 分析思维入门——从"取数"到"分析"

> 难度：⭐⭐ | 今日目标：建立SQL分析思维，理解SQL不仅是"取数工具"，更是"分析语言"
> 配对课程：01python_learn W5D1（时间日期处理）
> 前置依赖：01python_learn W6 将学SQL基础语法，本周先建立分析思维
> 扩展阅读：02analyst_thinking/week_01/day_01（SQL窗口函数）
> 面试地位：大厂数据分析面试必问SQL思维，约占面试评估的20%

---

## 一、核心概念与第一性原理

### 1.1 SQL的两层境界：取数 vs 分析

SQL的学习存在两个明显的层次。大多数人停留在第一层，而数据分析师必须达到第二层。

**第一层：取数** —— "帮我拉一下上个月的订单数据"
```sql
SELECT * FROM orders WHERE dt BETWEEN '2026-01-01' AND '2026-01-31';
```
这一层的特征是：需求方怎么说你就怎么写，输出的是"数据"，不是"答案"。你把Excel导出给对方，对方自己再去做分析。

**第二层：分析** —— "上个月哪个品类的复购率最高？"
```sql
SELECT category,
       COUNT(DISTINCT CASE WHEN order_cnt >= 2 THEN user_id END) * 1.0
         / COUNT(DISTINCT user_id) AS repurchase_rate
FROM (
    SELECT user_id, category, COUNT(DISTINCT order_id) AS order_cnt
    FROM orders
    WHERE dt BETWEEN '2026-01-01' AND '2026-01-31'
    GROUP BY user_id, category
)
GROUP BY category
ORDER BY repurchase_rate DESC;
```
第二层的特征是：你把业务问题翻译成数据问题，设计SQL逻辑，输出的直接就是"答案"。对方拿到这个结果可以直接做决策。

**从第一层到第二层的关键跃迁**：不是在"写SQL"这一步用功，而是在"想清楚要算什么"这一步用功。

### 1.2 SQL分析思维的五个阶段

一个完整的SQL分析流程包含五个环节，每一步都很重要：

```
业务问题 → 翻译成数据问题 → 设计SQL逻辑 → 写SQL → 验证结果 → 解读
```

| 阶段 | 做什么 | 常见错误 |
|------|-------|---------|
| 1. 理解业务问题 | 听清楚对方真正想要什么 | 对方说"拉个数据"，你就直接拉 |
| 2. 翻译成数据问题 | 把业务语言变成数据语言 | 没想清楚指标定义就动手写 |
| 3. 设计SQL逻辑 | 在纸上/脑里画出数据流 | 跳过这一步直接写SQL |
| 4. 写SQL | 把设计翻译成代码 | 写完后不验证，直接交结果 |
| 5. 验证与解读 | 检查结果是否合理，给出业务解读 | 只给数字不给解读 |

举个例子——运营说"帮我看看用户质量怎么样"：

| 阶段 | 你的操作 |
|------|---------|
| 理解问题 | "你说的'用户质量'是指留存、付费、还是活跃度？"→ 对方说"付费转化" |
| 翻译 | 付费转化率 = 付费用户数 / 活跃用户数（过去30天） |
| 设计逻辑 | ① 过去30天活跃用户 → ② 其中付费的 → ③ 按渠道/日期拆 |
| 写SQL | 用CTE+LEFT JOIN实现上述逻辑 |
| 验证解读 | 付费转化率12%，比上月提升2ppt，主要来自搜索渠道的优化 |

### 1.3 为什么SQL是数据分析师的母语

SQL之所以成为数据分析师的必备技能，有四个根本原因：

**1. 数据在哪里，SQL就在哪里**
- 几乎所有企业的数据都存储在关系型数据库中
- 数据仓库（Hive、Spark SQL、BigQuery）都使用SQL作为查询语言
- Python/Pandas能做的事，SQL在数据库层面能更高效地完成

**2. SQL是声明式语言——你说"要什么"，不说"怎么做"**
```sql
SELECT city, SUM(amount) FROM orders GROUP BY city;
```
你不用告诉数据库"怎么遍历、怎么分组、怎么累加"——只需要声明"按城市分组求金额总和"。数据库优化器会自动选择最优执行方案。

**3. SQL的逻辑可被优化器自动优化**
- 你写的SQL是逻辑描述，不是执行指令
- 数据库优化器会根据统计信息选择索引、JOIN顺序、并行度
- 同样的SQL，数据量增长100倍后优化器会自动调整策略

**4. SQL是团队协作的通用语言**
- 数据分析师写SQL提取指标
- 数据工程师写SQL做ETL
- 后端工程师写SQL做报表
- 产品经理也能看懂简单的SQL查询

### 1.4 SQL的执行顺序 vs 书写顺序——最易忽略的陷阱

你写的顺序和数据库执行的顺序完全不同，这是很多SQL错误的根源：

```
书写顺序：SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
执行顺序：FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

**这意味着什么？**

1. **你不能在WHERE里用SELECT里定义的别名**：因为WHERE在SELECT之前执行
```sql
-- 错误：total是SELECT里定义的别名，WHERE看不到
SELECT user_id, SUM(amount) AS total FROM orders WHERE total > 100 GROUP BY user_id;

-- 正确：用HAVING代替
SELECT user_id, SUM(amount) AS total FROM orders GROUP BY user_id HAVING total > 100;
```

2. **ORDER BY可以使用SELECT的别名**：因为ORDER BY在SELECT之后执行
```sql
-- 正确：ORDER BY能看到total
SELECT user_id, SUM(amount) AS total FROM orders GROUP BY user_id ORDER BY total DESC;
```

3. **LIMIT是最后执行的**：不影响前面的聚合计算

**完整执行顺序详解**：

| 步骤 | 子句 | 作用 | 此时数据状态 |
|------|------|------|-------------|
| 1 | FROM + JOIN | 确定数据源，执行JOIN | 原始表被连接，产生中间结果集 |
| 2 | ON | JOIN的匹配条件 | 决定哪些行匹配成功 |
| 3 | WHERE | 过滤行 | 不满足条件的行被丢弃 |
| 4 | GROUP BY | 分组 | 数据被分组，每个组一行 |
| 5 | HAVING | 过滤组 | 不满足条件的组被丢弃 |
| 6 | SELECT | 选择列、计算表达式 | 此时才能使用别名 |
| 7 | DISTINCT | 去重 | 移除重复行 |
| 8 | ORDER BY | 排序 | 最终排序 |
| 9 | LIMIT/OFFSET | 限制行数 | 截断结果集 |

### 1.5 分析型SQL的三个核心操作

几乎所有的业务分析SQL都由三类操作组合而成：

| 操作类型 | SQL语法 | 分析含义 | 示例 |
|---------|---------|---------|------|
| 聚合 | GROUP BY + SUM/COUNT/AVG | "总量是多少" | 每品类GMV |
| 过滤 | WHERE / HAVING | "只看符合条件的" | 只看今年数据 |
| 变换 | CASE WHEN / 窗口函数 | "把原始数据变成可分析的" | 把金额变成"高中低"分档 |

这三类操作的任意组合，可以解决80%以上的业务分析问题。剩下的20%需要用更高级的工具（如窗口函数、CTE、自连接）——这正是本周接下来要学的内容。

### 1.6 面试高频考点

**面试官常问**：
- "拿到一个分析需求，你会怎么入手？"
- "SQL的执行顺序是什么？为什么这个顺序很重要？"
- "你平时怎么保证SQL结果的准确性？"
- "一个SQL跑得很慢，你会怎么排查？"

**回答要点**：
- 先说"先理解业务问题→翻译→设计逻辑→写SQL→验证"，展现结构化思维
- 执行顺序：FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY→LIMIT
- 验证方法：抽样检查、与已知总量对比、让业务方确认逻辑
- 慢查询：先EXPLAIN看执行计划，看type是否为ALL，是否走了索引

---

## 二、知识点详细拆解

### 2.1 写SQL之前的"三步设计法"

**为什么重要**：直接写SQL容易陷入细节，三步设计法让你先想清楚全局再动手。

**第一步：定义输出粒度**
- 最终结果表里，**一行代表什么**？
- 是一个用户？一个品类×一天？一个订单？
- 这个决定影响GROUP BY的列、JOIN的方式、计数的逻辑

**第二步：确定数据源和关联方式**
- 需要哪些表？
- 怎么JOIN？INNER还是LEFT？
- JOIN键是什么？会不会导致行数膨胀？

**第三步：设计变换逻辑**
- 哪些列需要CASE WHEN转换？
- 哪些需要窗口函数（排名、累计、环比）？
- 哪些需要聚合（SUM、COUNT、AVG）？

**实例演示**：需求是"过去30天，各品类消费金额Top5的用户是谁？每个用户的消费金额、订单数、排名"

| 步骤 | 分析 | 结果 |
|------|------|------|
| 输出粒度 | 一个用户×一个品类 | GROUP BY user_id, category |
| 数据源 | orders表，过滤最近30天 | WHERE dt >= 30天前 |
| 变换逻辑 | 聚合金额和订单数 → 排名 → 取Top5 | SUM + COUNT + ROW_NUMBER + WHERE rn<=5 |

### 2.2 数据粒度的概念——一行代表什么

**为什么重要**：粒度的混淆是SQL错误最常见的根源。

数据的粒度决定了你看到的信息的精细程度：
- **订单粒度**：一行一个订单，可以看到每笔订单的详情
- **用户粒度**：一行一个用户，订单信息被聚合（SUM/COUNT）
- **用户×品类粒度**：一行是"某个用户在某个品类的消费汇总"
- **日期粒度**：一行一天，所有用户和订单被聚合

**粒度转换的核心操作**：
- 变粗（聚合）：GROUP BY → 行数减少，信息丢失
- 变细（展开）：JOIN → 行数增加，但带来更多维度
- 保持粒度：窗口函数 → 行数不变，新增计算列

**常见错误**：
- 在用户粒度上统计订单金额的AVG——应该是SUM（一个用户的平均订单金额 vs 一个用户的订单总金额）
- 在订单粒度上统计用户数——应该COUNT(DISTINCT user_id)，不是COUNT(*)
- JOIN导致行数膨胀后直接用COUNT(*)——应该COUNT(DISTINCT)

### 2.3 COUNT的高级用法

**为什么重要**：COUNT是最常用的聚合函数，但其变体经常被用错。

| 写法 | 含义 | 注意事项 |
|------|------|---------|
| COUNT(*) | 统计行数 | 包括NULL行 |
| COUNT(column) | 统计该列非NULL的行数 | NULL不计入 |
| COUNT(DISTINCT column) | 统计该列去重后的非NULL值数 | 性能较差，大表慎用 |
| COUNT(DISTINCT CASE WHEN ... THEN column END) | 条件去重计数 | 最强大但也最容易被忽略 |

**例子**：统计"过去30天至少有一笔订单金额大于500的用户数"

```sql
-- 正确写法
SELECT COUNT(DISTINCT CASE WHEN amount > 500 THEN user_id END) AS high_value_users
FROM orders
WHERE dt >= DATE('now', '-30 days');

-- 如果写成以下任何一种都会出错：
-- COUNT(*) → 统计了订单数，不是用户数
-- COUNT(DISTINCT user_id) → 没有筛选金额条件
-- COUNT(user_id) WHERE amount > 500 → 语法错误
```

### 2.4 CASE WHEN的五种分析模式

**为什么重要**：CASE WHEN是SQL中最灵活的数据变换工具，掌握它的五种模式能大幅提升分析效率。

**模式1：简单二分类**
```sql
CASE WHEN amount > 100 THEN '高' ELSE '低' END
```

**模式2：多分类**
```sql
CASE WHEN amount > 500 THEN '高价值'
     WHEN amount > 200 THEN '中价值'
     WHEN amount > 50 THEN '低价值'
     ELSE '无消费' END
```

**模式3：聚合中的条件计数**
```sql
COUNT(DISTINCT CASE WHEN status='paid' THEN user_id END) AS paid_users
```

**模式4：聚合中的条件求和**
```sql
SUM(CASE WHEN category='数码' THEN amount ELSE 0 END) AS digital_gmv
```

**模式5：行转列（PIVOT）**
```sql
SELECT user_id,
       SUM(CASE WHEN category='服装' THEN amount ELSE 0 END) AS clothing,
       SUM(CASE WHEN category='数码' THEN amount ELSE 0 END) AS digital,
       SUM(CASE WHEN category='食品' THEN amount ELSE 0 END) AS food
FROM orders GROUP BY user_id;
```

### 2.5 NULL的处理哲学

**为什么重要**：NULL是SQL中最容易出错的概念之一，因为它不是值，而是"未知"。

**NULL的核心特性**：
- 任何值和NULL比较都返回UNKNOWN（不是TRUE，也不是FALSE）
- NULL = NULL 返回 UNKNOWN（不是TRUE！）
- 聚合函数（除COUNT(*)）自动忽略NULL
- COUNT(column)不计算NULL行

**处理NULL的标准工具**：
| 函数 | 作用 | 示例 |
|------|------|------|
| COALESCE(val, default) | 返回第一个非NULL值 | COALESCE(amount, 0) |
| IFNULL(val, default) | COALESCE的简化版（部分数据库） | IFNULL(amount, 0) |
| NULLIF(a, b) | 如果a=b返回NULL，否则返回a | NULLIF(amount, 0) |
| IS NULL / IS NOT NULL | 判断是否为NULL | WHERE amount IS NULL |

### 2.6 本周学习地图

```
周一：SQL分析思维入门 ← 你在这里
   ↓ 建立分析思维框架
周二：窗口函数——聚合不折叠
   ↓ 掌握SQL最强大的分析工具
周三：连续登录与留存——Gap-and-Island
   ↓ 解决连续行为分析问题
周四：JOIN全家桶——不只是LEFT JOIN
   ↓ 避免最隐蔽的数据错误
周五：CTE与子查询——让复杂SQL可读
   ↓ 写出可维护的复杂SQL
周六：TopN/复购率/漏斗——经典业务SQL
   ↓ 覆盖70%的日常分析需求
周日：周复习——综合实战
```

> **今天最重要的认知**：SQL不是"取数工具"，而是"分析语言"。同样的数据，取数思维写出 `SELECT *`，分析思维写出一个有逻辑、能回答业务问题的查询。区别在于你在写SQL之前，脑子里有没有先想清楚"我要回答什么问题、数据需要怎么变换"。

---

## 三、动手练习

### 练习0：环境准备
使用Python的sqlite3创建模拟数据，用于所有练习。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 创建订单表
n = 200
df = pd.DataFrame({
    'order_id': range(1, n+1),
    'user_id': np.random.choice(range(1, 31), n),
    'category': np.random.choice(['服装', '数码', '食品', '美妆'], n),
    'amount': np.round(np.random.lognormal(4, 0.8, n), 2),
    'order_date': np.random.choice(pd.date_range('2026-01-01', '2026-01-31'), n),
    'status': np.random.choice(['paid', 'unpaid', 'refunded'], n, p=[0.7, 0.2, 0.1])
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

print(f"已创建 {n} 行订单数据")
print(f"涉及 {df['user_id'].nunique()} 个用户，{df['category'].nunique()} 个品类")
print(f"日期范围: {df['order_date'].min()} ~ {df['order_date'].max()}")
print()
print("前5行预览：")
print(df.head())

### 练习1：区分"取数SQL"和"分析SQL"

In [ ]:
# ===== 练习1：取数 vs 分析 =====

sql_examples = [
    ("SELECT * FROM orders WHERE order_date='2026-01-15'", "取数",
     "需求方说"拉一下1月15号的订单"，你直接SELECT *，这是取数"),
    ("""SELECT category, COUNT(DISTINCT user_id) AS buyers,
       SUM(amount) AS gmv FROM orders WHERE order_date>='2026-01-01'
       GROUP BY category ORDER BY gmv DESC""", "分析",
     "你主动按品类汇总GMV和买家数，输出了"答案"而非"数据""),
    ("""SELECT user_id, COUNT(*) AS order_cnt,
       CASE WHEN COUNT(*)>=3 THEN '高频' ELSE '低频' END AS freq_label
       FROM orders GROUP BY user_id""", "分析",
     "不仅统计了订单数，还做了分群变换——这正是分析型SQL的特征"),
    ("""SELECT * FROM orders WHERE user_id IN (1,2,3) ORDER BY order_date""", "取数",
     "只是把特定用户的订单取出来，没有做任何聚合或变换"),
]

for sql, answer, reason in sql_examples:
    print(f"[{answer}] {reason}")
    print(sql[:100] + "..." if len(sql) > 100 else sql)
    print()

print("关键区分标准：")
print("  取数SQL：SELECT * + 简单WHERE → 输出原始数据")
print("  分析SQL：GROUP BY + 聚合函数 + CASE WHEN + 窗口函数 → 输出答案")
print("  有GROUP BY不一定就是分析——关键看是否回答了业务问题")

### 练习2：SQL逻辑设计（三步法练习）

In [ ]:
# ===== 练习2：三步设计法 =====

print("=" * 60)
print("场景1：找出'过去30天，消费金额Top10的用户'")
print("=" * 60)
print()
print("三步设计：")
print("  1. 输出粒度：一行 = 一个用户")
print("     输出列：user_id, total_amount(SUM), order_cnt(COUNT)")
print("  2. 数据源：orders表（WHERE过滤30天内）")
print("  3. 变换逻辑：GROUP BY user_id → ORDER BY total_amount DESC → LIMIT 10")
print()

# 写SQL
q2a = """SELECT user_id, SUM(amount) AS total_amount, COUNT(*) AS order_cnt
FROM orders
WHERE order_date >= '2026-01-01' AND order_date < '2026-02-01'
GROUP BY user_id
ORDER BY total_amount DESC
LIMIT 10"""
print("SQL执行结果：")
print(pd.read_sql(q2a, conn))

print()
print("=" * 60)
print("场景2：找出'各品类消费最高的用户是谁'")
print("=" * 60)
print()
print("三步设计：")
print("  1. 输出粒度：一行 = 一个品类 + 一个用户")
print("     输出列：category, user_id, total_amount, rank")
print("  2. 数据源：orders表")
print("  3. 变换逻辑：先GROUP BY category, user_id → ROW_NUMBER排名 → 取rn=1")
print()

q2b = """SELECT category, user_id, total_amount
FROM (
    SELECT category, user_id, SUM(amount) AS total_amount,
           ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn
    FROM orders
    GROUP BY category, user_id
)
WHERE rn = 1
ORDER BY total_amount DESC"""
print("SQL执行结果：")
print(pd.read_sql(q2b, conn))

print()
print("对比两个场景：粒度不同 → GROUP BY不同 → 输出含义完全不同")
print("这就是'三步设计法'的价值——在写SQL之前先想清楚粒度")

### 练习3：验证SQL执行顺序

In [ ]:
# ===== 练习3：验证执行顺序陷阱 =====

print("验证1：WHERE中不能使用SELECT别名")
try:
    pd.read_sql("""SELECT user_id, SUM(amount) AS total
        FROM orders WHERE total > 500 GROUP BY user_id""", conn)
except Exception as e:
    print(f"  报错: {e}")
    print("  原因：WHERE在SELECT之前执行，不认识别名'total'")

print()
print("验证2：HAVING中可以使用聚合结果")
q3 = """SELECT user_id, SUM(amount) AS total
FROM orders
GROUP BY user_id
HAVING total > 500
ORDER BY total DESC"""
print(pd.read_sql(q3, conn))
print("  HAVING在GROUP BY之后执行，可以使用聚合结果")

print()
print("验证3：ORDER BY可以使用SELECT别名")
q3b = """SELECT user_id, SUM(amount) AS total
FROM orders
GROUP BY user_id
ORDER BY total DESC
LIMIT 5"""
print(pd.read_sql(q3b, conn))
print("  ORDER BY在SELECT之后执行，可以使用别名'total'")

### 练习4：COUNT和CASE WHEN的综合练习

In [ ]:
# ===== 练习4：COUNT的四种写法对比 =====

print("统计各品类'已支付订单'的数量和金额，同时统计'高价值订单(>300)'的数量")
print()

q4 = """SELECT
    category,
    COUNT(*) AS total_orders,
    COUNT(CASE WHEN status='paid' THEN 1 END) AS paid_orders,
    SUM(CASE WHEN status='paid' THEN amount ELSE 0 END) AS paid_amount,
    COUNT(DISTINCT CASE WHEN amount > 300 THEN user_id END) AS high_value_users,
    ROUND(
        SUM(CASE WHEN status='paid' THEN amount ELSE 0 END) * 1.0 / NULLIF(SUM(amount), 0), 3
    ) AS paid_ratio
FROM orders
GROUP BY category
ORDER BY total_orders DESC"""
print(pd.read_sql(q4, conn))

print()
print("注意：")
print("  - COUNT(*) 统计所有行")
print("  - COUNT(CASE WHEN ... END) 只统计满足条件的行")
print("  - COUNT(DISTINCT CASE WHEN ... END) 去重后统计")
print("  - NULLIF(amount, 0) 避免除零错误")

---

## 四、业务场景应用

### 场景1：从"帮我拉个数据"到"帮我回答个问题"

运营同事在群里说："帮我拉一下昨天的订单明细。"

**取数思维的做法**：
```sql
SELECT * FROM orders WHERE dt='2026-01-15'
```
导出Excel发给对方。10分钟后对方又来："再帮我拉一下上周同期的。"你继续拉。30分钟后对方又来了："再帮我拉一下上个月同期的。"

**分析思维的做法**：
"你是想看周同比变化吗？要不要我直接帮你算出来——昨天的GMV、上周同期的GMV、变化率、按品类拆、看是哪个品类涨了哪个跌了？"

运营："对对对就是这个！"

你一次输出：

| 品类 | 昨天GMV | 上周同期 | 环比变化 | 上月同期 | 同比变化 |
|------|---------|---------|---------|---------|--------|
| 服装 | 125,000 | 118,000 | +5.9% | 98,000 | +27.6% |
| 数码 | 89,000 | 92,000 | -3.3% | 85,000 | +4.7% |
| 食品 | 67,000 | 64,000 | +4.7% | 58,000 | +15.5% |

**区别在于：前者看到的是"取数需求"，后者看到的是"分析需求"。一个分析师的价值，在于把前者变成后者。**

### 场景2：业务方说"用户质量"下降——你怎么定位？

老板说"最近用户质量不太好"，你需要把这个模糊的判断变成可量化的分析。

**分析路线**：
1. 定义"用户质量"：是留存降了？复购率降了？客单价降了？还是转化率降了？
2. 拉出最近4周的指标趋势
3. 如果有下降，按渠道/地区/品类拆解
4. 定位到具体维度后，深入看该维度发生了什么变化

**SQL逻辑框架**：
```sql
-- 步骤1：周度指标趋势
WITH weekly_metrics AS (
    SELECT
        strftime('%Y-W%W', order_date) AS week,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(DISTINCT CASE WHEN order_cnt >= 2 THEN user_id END) * 1.0
            / COUNT(DISTINCT user_id) AS repurchase_rate,
        AVG(amount) AS avg_order_value
    FROM (
        SELECT user_id, order_date, amount,
               COUNT(*) OVER(PARTITION BY user_id, strftime('%Y-W%W', order_date)) AS order_cnt
        FROM orders
    )
    GROUP BY week
)
SELECT * FROM weekly_metrics ORDER BY week DESC LIMIT 8;

-- 步骤2：如果复购率下降，按渠道拆解
-- (如果有渠道字段)

-- 步骤3：定位到具体渠道后，看该渠道的用户画像是否变化
```

### 场景3：日报自动化的SQL设计

每天早上9点，需要自动生成前一天的运营日报。这个需求考察的是你能否设计一个"稳定、准确、高效"的SQL。

**设计要点**：
- 日期参数化：用变量或日期函数而非硬编码
- 核心指标：GMV、订单数、新增用户、活跃用户、复购率
- 同比对比：昨天 vs 上周同期 vs 上月同期
- 异常检测：如果某个指标偏离正常范围，自动标记

**SQL框架**：
```sql
-- 日报SQL模板
WITH yesterday AS (
    SELECT '昨天' AS period,
           COUNT(DISTINCT user_id) AS buyers,
           SUM(amount) AS gmv,
           COUNT(*) AS orders
    FROM orders
    WHERE order_date = DATE('now', '-1 day')
),
last_week AS (
    SELECT '上周同期' AS period,
           COUNT(DISTINCT user_id) AS buyers,
           SUM(amount) AS gmv,
           COUNT(*) AS orders
    FROM orders
    WHERE order_date = DATE('now', '-8 days')
),
last_month AS (
    SELECT '上月同期' AS period,
           COUNT(DISTINCT user_id) AS buyers,
           SUM(amount) AS gmv,
           COUNT(*) AS orders
    FROM orders
    WHERE order_date = DATE('now', '-31 days')
)
SELECT * FROM yesterday
UNION ALL SELECT * FROM last_week
UNION ALL SELECT * FROM last_month;
```

### 场景4：数据验真——你的SQL结果对吗？

写完SQL之后，怎么确认结果是对的？

**验真方法**：
1. **总量校验**：有没有所有用户的总GMV和从业务系统里拿到的总GMV对得上？
2. **行数校验**：输出行数是否和预期一致？如果GROUP BY了10个品类，应该有10行
3. **边界校验**：看最大值和最小值是否在合理范围内（金额有没有负数？）
4. **抽样校验**：随机挑一个用户，手动数他的订单，和SQL结果对比
5. **NULL检查**：有没有不该为NULL的列为NULL了？

---

## 五、分级检验清单

### 初级检验（必须掌握）
- [ ] 能区分"取数SQL"和"分析SQL"
- [ ] 能说出SQL的执行顺序（FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY）
- [ ] 能解释为什么WHERE中不能用SELECT别名
- [ ] 拿到分析需求时，会先"设计逻辑"再"写SQL"

**答案提示**：
- 取数SQL输出原始数据，分析SQL输出答案
- 执行顺序记忆口诀："FWGHSOL"——From→Where→Group→Having→Select→Order→Limit
- WHERE在SELECT之前执行，因此看不到别名
- 三步设计法：粒度→数据源→变换逻辑

### 中级检验（能解释给同事听）
- [ ] 能把一个业务问题翻译成三步设计
- [ ] 能判断一个SQL查询的输出粒度（一行代表什么）
- [ ] 能区分COUNT(*)、COUNT(column)、COUNT(DISTINCT)的使用场景
- [ ] 能在脑子里预演SQL的大致结果

**答案提示**：
- 粒度 = GROUP BY的列 = 输出的每行代表什么
- COUNT(*)算行数，COUNT(column)算非NULL值数，COUNT(DISTINCT)去重
- 预演：先想GROUP BY会产生多少行，再想聚合函数会产生什么值

### 高级检验（能举一反三）
- [ ] 能把复杂的多步骤分析拆成多个简单的SQL
- [ ] 能从业务方的"取数需求"中识别出真正的分析需求
- [ ] 能评估一个SQL的性能（扫描量、JOIN大小、倾斜风险）
- [ ] 能设计自动化的日/周/月报SQL

**答案提示**：
- 复杂分析拆解：每一步一个CTE，逐步计算
- 取数→分析：追问"你拿了这个数据想干什么"→把目的转成SQL
- 性能评估：看WHERE过滤后的数据量、JOIN的键是否有索引、是否有全表扫描

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 01 W4** | 数仓分层 | 你的SQL跑在DWD还是DWS层？不同层的粒度不同 |
| **← Python** | Pandas数据分析 | Pandas的groupby和SQL的GROUP BY是同一套思维 |
| **→ 明日** | 窗口函数 | SQL分析最强大的工具——聚合不折叠 |
| **→ 本周三** | 连续登录 | gap-and-island：把连续变成相等 |
| **→ 本周六** | TopN/复购率/漏斗 | 三道经典分析SQL——日常工作的70% |
| **→ 01 W6** | SQL基础语法 | 下周在01里系统学SQL语法，本周先建立分析思维 |
| **扩展阅读** | 02analyst_thinking W1D1 | 窗口函数的完整知识图谱和面试题 |

### 横向连接：分析思维的通用性

SQL的分析思维不只适用于SQL——同样的思维方式在Pandas、Excel、BI工具中都是通用的：

| SQL概念 | Pandas对等 | Excel对等 |
|---------|-----------|----------|
| WHERE | df[df['col'] > 100] | 筛选功能 |
| GROUP BY | df.groupby('col') | 数据透视表 |
| CASE WHEN | np.where() | IF函数 |
| 窗口函数 | df['col'].rolling(7) | 滚动平均 |
| JOIN | pd.merge() | VLOOKUP |
| CTE | 中间变量赋值 | 辅助列 |

---

## 七、当日小结

> **今天最重要的一个认知**：写SQL之前先用30秒想清楚三件事——输出的一行代表什么？从哪些表取？数据需要怎么变换？这30秒帮你省下后面30分钟的重写和调试。

**三条金句带回家**：
1. SQL有两层境界——取数层输出数据，分析层输出答案。你拿的是分析师工资，就要做第二层的事
2. SQL的执行顺序是FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY→LIMIT——这个顺序是理解一切SQL行为的基础
3. 拿到需求先设计后写——三步法（粒度→数据源→变换）比直接写SQL快三倍

### 课后拓展

1. **练习**：找一个你工作中常见的"取数需求"，尝试把它升级为"分析需求"，写出相应的SQL逻辑设计

2. **思考**：如果订单表有10亿行，你在设计SQL时需要考虑哪些额外因素？（提示：分区、索引、采样）

3. **预习**：明天要学的窗口函数是本周最重要的工具——今晚可以先想一个问题：GROUP BY折叠了行，但很多场景下我们需要"既不折叠行，又要参考其他行做计算"。怎么解决这个矛盾？

4. **面试准备**：背熟以下问题——"拿到一个分析需求，你的完整流程是什么？""SQL的执行顺序是什么？""NULL在SQL中的特殊行为？"

---

*03analyst_thinking · Week 5 Day 1 · 2026-05-21*